In [1]:
"""
Trains on ml_feature_table.csv (v2 feature set).
Self-contained: run this whole file/cell top to bottom in one go.
"""

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

FEATURE_COLUMNS = [
    "task_weight", "planned_task_duration_days", "is_planned", "is_cross_department",
    "major_activity_weight", "pct_of_planned_duration_elapsed", "days_remaining_at_prediction",
    "started_late", "num_subtasks_as_of_prediction", "subtask_completion_pct_as_of_prediction",
    "has_subtasks_at_prediction", "num_revisions_before_prediction",
    "position_historical_overdue_rate", "position_has_history",
    "employee_active_workload_at_prediction",
    "department_historical_overdue_rate", "department_recent_overdue_rate_30d",
    "department_has_history",
]
BOOL_COLUMNS = ["is_planned", "is_cross_department", "started_late",
                 "has_subtasks_at_prediction", "position_has_history", "department_has_history"]
BOOL_MAP = {"t": True, "f": False, "true": True, "false": False}


def to_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().map(BOOL_MAP).astype(bool)


# ---- Load ----
df = pd.read_csv("ml_feature_table_for_1_day_after.csv")

# ---- Convert booleans BEFORE anything else touches them ----
for col in BOOL_COLUMNS:
    if col in df.columns:
        df[col] = to_bool(df[col])
if df["target"].dtype != bool:
    df["target"] = to_bool(df["target"])

# Hard stop if conversion didn't work, instead of failing later inside sklearn
for col in BOOL_COLUMNS + ["target"]:
    assert df[col].dtype == bool, f"{col} is still {df[col].dtype}, not bool -- fix before training"

if "prediction_date" in df.columns:
    df["prediction_date"] = pd.to_datetime(df["prediction_date"])

# ---- Time-based split ----
df_sorted = df.sort_values("prediction_date").reset_index(drop=True) if "prediction_date" in df.columns else df
split_idx = int(len(df_sorted) * 0.8)
train_df, test_df = df_sorted.iloc[:split_idx], df_sorted.iloc[split_idx:]
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# ---- Build X/y. Booleans -> int (0/1) so sklearn's float32 cast never sees a string ----
X_train = train_df[FEATURE_COLUMNS].apply(lambda c: c.astype(int) if c.dtype == bool else c).fillna(-1)
y_train = train_df["target"].astype(int)
X_test = test_df[FEATURE_COLUMNS].apply(lambda c: c.astype(int) if c.dtype == bool else c).fillna(-1)
y_test = test_df["target"].astype(int)

# Final safety check: every column must now be numeric
non_numeric = [c for c in X_train.columns if not pd.api.types.is_numeric_dtype(X_train[c])]
assert not non_numeric, f"Still non-numeric columns going into the model: {non_numeric}"

model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_proba):.3f}")

print("\nFeature importances:")
for name, imp in sorted(zip(FEATURE_COLUMNS, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.3f}")

joblib.dump(model, "overdue_risk_model_v2_for_1_day_after.pkl")
test_df.assign(predicted_proba=y_proba).to_csv("test_predictions_v2_for_1_day_after.csv", index=False)
print("\nSaved overdue_risk_model_v2.pkl and test_predictions_v2.csv")

Train: 10539, Test: 2635
              precision    recall  f1-score   support

           0       0.89      0.82      0.85      2113
           1       0.45      0.61      0.52       522

    accuracy                           0.78      2635
   macro avg       0.67      0.71      0.69      2635
weighted avg       0.81      0.78      0.79      2635

ROC-AUC: 0.779
PR-AUC:  0.534

Feature importances:
  department_recent_overdue_rate_30d: 0.272
  position_historical_overdue_rate: 0.177
  department_historical_overdue_rate: 0.127
  employee_active_workload_at_prediction: 0.091
  num_revisions_before_prediction: 0.064
  planned_task_duration_days: 0.048
  days_remaining_at_prediction: 0.047
  pct_of_planned_duration_elapsed: 0.042
  task_weight: 0.041
  major_activity_weight: 0.038
  is_planned: 0.023
  started_late: 0.018
  position_has_history: 0.005
  department_has_history: 0.005
  is_cross_department: 0.002
  num_subtasks_as_of_prediction: 0.000
  subtask_completion_pct_as_of_predict